# E010: twin-business and name-ambiguity features (+ optional dense similarity, diagnostics)

E009 = v4 features + stage-2 stacking + per-S1 expected-F0.5 selection with an explicit "predict nothing"
option + count matching + France normalization. E010 keeps all of that.

**Error analysis (train, ground truth joined to the out-of-fold errors):** most false merges are synthetic
**twin businesses** that belong to no S1: a copy of a real business with a nudged house/unit number
(2921 -> 2942, C-4/32 -> C-4/36, Unit 24 -> 26) and often one extra name word (Foods, Infratech, Wireless).
Every record of the twin carries the same change, so the twin forms its own group. Number noise also hits
true records (1451 -> 3451), so the discriminator is the group, not the number. Second bucket: records with
an empty address whose name is carried by several S1 entities (Lucky Grill, Jai Software).

| new | what |
|---|---|
| stage 2 group consensus | each S1's candidates grouped by exact address, core name, house number: group size, probability mass without the pair, share of the S1's mass, gap to the heaviest group |
| stage 1 name ambiguity | # S1 entities sharing the S1's core name, # S1 entities carrying the record's core name (rivals outside the candidate list too), # S2/S3 records with that name |
| optional dense | `DENSE="e5s"`: multilingual-e5-small cosines (needs GPU, +1-1.5 h) |

| plan item | E010 status |
|---|---|
| A. dense similarity as features | optional (`DENSE`): `intfloat/multilingual-e5-small` (MIT, 118M) cosine of the raw name (`d_name`) and of "name, address" (`d_full`) for **every** candidate pair, plus rank/gap among the S1's candidates and among the S2/S3 record's competing S1s. Raw text, so native-script names reach the model unchanged |
| B. hard negatives | already the training set: every training pair is a blocker candidate (~60 per S1), i.e. only mined hard negatives, including common-name crowds and same-address different-entity pairs (v4 crowd features). Extra re-weighting waits for the E009 false-positive buckets |
| C. hybrid matcher | LightGBM over lexical + char + BM25/conj + address + **dense** + rank/gap + competition/context |
| D. calibration / abstention | kept from E009 (isotonic p, per-S1 expected F0.5, empty option) |
| E. script-aware diagnostics | **new**: CV macro F0.5 per slice: true match in native script vs Latin only, singletons, cluster size, S1 sharing its exact address with another S1 |
| F. candidate expansion | after the E009/E010 error buckets |

**Run in the SAME Kaggle notebook as E009** (all caches reused: prepared data, candidates, v2/v3/v4 feature
matrices). With `DENSE="none"` no GPU is needed; with `DENSE="e5s"` switch the accelerator to GPU
(Internet On for the model download). Persistence Files. *Run all* resumes.

In [ ]:
# 1. Config
EXP       = "20260927-E010-twins"
E009_EXP  = "20260926-E009-v4"
DENSE     = "none"   # (GPU still needed if candidates must be re-blocked: bge_native channel) # "none" (CPU only) | "e5s" = multilingual-e5-small (118M, GPU) | "e5b" (278M) | "bge" (bge-m3, 568M)
E007_SPEC = "base,conj:20,bm25:5,bge_native:5,name_noaddr:5"
CV_FRAC   = 0.1      # same S1 sample as E008/E009 -> cached v2/v3/v4 feature matrices are reused
LR        = 0.1
REPO      = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR  = "/kaggle/working/ber"
WORK      = "/kaggle/working/work"

In [ ]:
# 2. Dataset, validator, caches
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found"
DATA = os.path.dirname(os.path.dirname(hits[0]))
val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
VALIDATOR = val[0] if val else None
def reuse(pattern, dest_dir):
    for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        d = os.path.join(dest_dir, os.path.basename(p))
        if not os.path.exists(d):
            os.makedirs(dest_dir, exist_ok=True); os.symlink(p, d); print("reusing", p)
for split in ("train", "test"):
    reuse(f"work/prepared/{split}/*", f"{WORK}/prepared/{split}")
    reuse(f"work/{split}/*", f"{WORK}/{split}")
print("DATA =", DATA, "| VALIDATOR =", VALIDATOR)
!ls -la {WORK}/train {WORK}/test; free -g; nproc; nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Code, dependencies, tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6 && (python -c "import sentence_transformers" 2>/dev/null || pip install -q -U sentence-transformers)
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# 2b. Disk cleanup (Output quota ~19.5 GiB). Keeps: prepared data, the E007 candidates, every E008/E009 feature
#     cache, E009 metrics + test probabilities, and ONE E009 submission (C = best LB 0.950, with its candidate file).
import shutil, subprocess
TAG = "ch-" + E007_SPEC.replace(",", "+").replace(":", "") + "_m0_df2500"
REBLOCK = []
KEEP_EXP = {"20260926-E009-v4", EXP}
drop = []
for d in ("output", "E004_results", "probe_E004", "reports_E004", "E008_results", "output_E009_A", "output_E009_B"):
    drop.append(f"/kaggle/working/{d}")
for split in ("train", "test"):
    # caches of any other blocker configuration (incl. the incomplete run without bge_native)
    drop += [p for p in glob.glob(f"{WORK}/{split}/*ch-*") if TAG not in p]
    # candidates missing -> they will be re-blocked, and GPU blocking need not reproduce the same rows:
    # every per-pair cache of this TAG would be silently misaligned, so it is rebuilt too
    if not os.path.exists(f"{WORK}/{split}/cand_{TAG}.parquet"):
        drop += [p for p in glob.glob(f"{WORK}/{split}/*{TAG}*")]
        REBLOCK.append(split)
drop += [p for p in glob.glob(f"{WORK}/experiments/*") if os.path.basename(p) not in KEEP_EXP | {"20260925-E008-E007"}]
drop += glob.glob(f"{WORK}/experiments/20260925-E008-E007/test_pairs_proba.parquet")
for p in drop:
    if os.path.islink(p) or os.path.isfile(p):
        print("rm", p); os.remove(p)
    elif os.path.isdir(p):
        print("rm -r", p); shutil.rmtree(p)
print("splits to re-block:", REBLOCK)
!du -sh /kaggle/working 2>/dev/null; du -sh /kaggle/working/* {WORK}/* {WORK}/train/* {WORK}/test/* 2>/dev/null | sort -h | tail -25

In [ ]:
# helper
import subprocess, time, json
import pandas as pd
def ber(cmd, *extra):
    args = ["python", "-m", "ber.run", cmd, "--data", DATA, "--work", WORK, "--exp", EXP, "--cv-frac", str(CV_FRAC),
            "--lr", str(LR), "--df-cap", "2500", "--channels", E007_SPEC, "--feat", "v5", "--dense-model", DENSE,
            *map(str, extra)]
    t = time.time()
    p = subprocess.Popen(args, cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"{cmd} failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- {cmd} done in {(time.time() - t) / 60:.1f} min")
X = lambda name: f"{WORK}/experiments/{EXP}/{name}"
cols = ["macro_f05", "micro_precision", "micro_recall", "f05_singletons", "f05_nonsingletons"]

In [ ]:
# 3b. Blocking (only when the E007 candidates are missing). Needs the GPU for the bge_native channel:
#     without it the channel is silently skipped and candidate recall drops (train 0.977 instead of 0.982).
if REBLOCK:
    import torch
    assert torch.cuda.is_available(), "set Accelerator = GPU T4: re-blocking needs it for bge_native"
    for split in ("train", "test"):
        if not os.path.exists(f"{WORK}/{split}/cand_{TAG}.parquet"):
            ber("block", "--split", split)
import pyarrow.parquet as pq
for split in ("train", "test"):
    print(split, "candidate pairs:", pq.ParquetFile(f"{WORK}/{split}/cand_{TAG}.parquet").metadata.num_rows)

In [ ]:
# 4. Optional dense cosines for every candidate pair (GPU; train 145M pairs, test 114M pairs; cached as float16 .npy)
from ber.normalize import NORM_VERSION
assert os.path.exists(f"{WORK}/prepared/test/NORM_V{NORM_VERSION}"), "run E009 cell 6 first (France test normalization)"
for split in ("train", "test") if DENSE != "none" else ():
    if not glob.glob(f"{WORK}/{split}/dense_{DENSE}_full_*.npy"):
        ber("dense", "--split", split)
    print(split, json.load(open(f"{WORK}/{split}/dense_{DENSE}_info.json")))

In [ ]:
# 5. Stage-1 CV with v5 (3 folds, selection study, stress + count matching, diagnostic slices)
if not os.path.exists(X("cv_metrics.json")):
    ber("cv", "--folds", 3)
cv = json.load(open(X("cv_metrics.json")))
e9 = json.load(open(f"{WORK}/experiments/{E009_EXP}/cv_metrics.json"))
keys = [k for k in cv if k.startswith(("cv", "stress")) and isinstance(cv[k], dict)]
display(pd.concat({"E010": pd.DataFrame({k: cv[k] for k in keys}).T[cols],
                   "E009": pd.DataFrame({k: e9[k] for k in keys if k in e9}).T[cols]}, axis=1).round(4))
print("threshold", cv["threshold"], "| best selection", cv["selection"]["best"], "->", round(cv["selection"]["macro_f05"], 5),
      "| E009:", round(e9["selection"]["macro_f05"], 5))
print("new stage-1 features", {k: v for k, v in cv["feature_gain_top"].items() if k.startswith(("d_", "l_name_s1", "r_name_s"))})
print("top features", list(cv["feature_gain_top"].items())[:12])
display(pd.DataFrame({k[6:]: cv[k] for k in cv if k.startswith("slice_")}).T[cols + ["n_s1"]].round(4))

In [ ]:
# 6. Stage 2 (stacking) + selection study
if not os.path.exists(X("stack_metrics.json")):
    ber("stack", "--folds", 3)
sm = json.load(open(X("stack_metrics.json")))
s9 = json.load(open(f"{WORK}/experiments/{E009_EXP}/stack_metrics.json"))
rows = [("E009 chosen", s9["chosen"]["macro_f05"]), ("E010 stage 1 best selection", sm["stage1"]["macro_f05"]),
        ("E010 stage 2 best selection", sm["stage2"]["macro_f05"]), ("E010 chosen", sm["chosen"]["macro_f05"])]
display(pd.DataFrame(rows, columns=["variant", "CV macro F0.5"]).round(5))
print("use_stack:", sm["use_stack"], "| rule:", {k: v for k, v in sm["rule"].items() if k != "cal"})
print("group features", {k: v for k, v in sm["stack_feature_gain_top"].items() if k.startswith("grp_")})
display(pd.DataFrame({k: sm[k] for k in sm if k.startswith(("chosen", "slice_"))}).T[cols + ["n_s1"]].round(4))

In [ ]:
# 7. Test prediction and three submissions: A plain, B count-match France only, C count-match every country
OUTS = {"A": ("none", "/kaggle/working/output_E010_A"), "B": ("unseen", "/kaggle/working/output_E010_B"),
        "C": ("all", "/kaggle/working/output_E010_C")}
if not os.path.exists(X("test_pairs_proba.parquet")):
    ber("predict", "--out", OUTS["A"][1])
for k, (cm, od) in OUTS.items():
    if not os.path.exists(f"{od}/matching_results.tsv"):
        ber("select", "--count-match", cm, "--out", od, *(["--cand-from", OUTS["A"][1]] if k != "A" else []))
    if VALIDATOR:
        !python {VALIDATOR} --matching {od}/matching_results.tsv --candidate {od}/candidate_pairs.tsv --test-dir {DATA}/test --check-ids | tail -2
    info = json.load(open(f"{od}/selection_info.json"))
    print(k, "count_match =", cm, "| delta", info["delta"])
    display(pd.DataFrame(info["per_country"]).T.round(3))

In [ ]:
# 8. Bundle: send /kaggle/working/E010_results.tgz back (metrics + error sample; no submissions inside)
import shutil, tarfile
B = "/kaggle/working/E010_results"
os.makedirs(B, exist_ok=True)
for f in ("cv_metrics.json", "stack_metrics.json", "oof_errors.parquet"):
    if os.path.exists(X(f)):
        shutil.copy(X(f), f"{B}/{f}")
for split in ("train", "test"):
    if os.path.exists(f"{WORK}/{split}/dense_{DENSE}_info.json"):
        shutil.copy(f"{WORK}/{split}/dense_{DENSE}_info.json", f"{B}/dense_info_{split}.json")
for k, (_, od) in OUTS.items():
    if os.path.exists(f"{od}/selection_info.json"):
        shutil.copy(f"{od}/selection_info.json", f"{B}/selection_info_{k}.json")
with tarfile.open("/kaggle/working/E010_results.tgz", "w:gz") as t:
    t.add(B, arcname="E010_results")
print(sorted(os.listdir(B)), os.path.getsize("/kaggle/working/E010_results.tgz") // 1024, "KiB")